# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Email delivery

## 1. Installation & Setup

!pip install -q openai pydantic gradio python-dotenv


## 2. Configuration

**Edit the values below before running the notebook.**

In [1]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
from dotenv import load_dotenv
import os

load_dotenv()

# API Keys
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Email settings (Gmail with App Password)
EMAIL_SENDER = os.getenv('EMAIL_SENDER', "bellareadsai@gmail.com")
EMAIL_PASSWORD = os.getenv('EMAIL_PASSWORD', "your-app-password")

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5


## 3. Pydantic Models for Validation

In [2]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. OpenAI Setup & Prompts

In [3]:
from openai import OpenAI

def get_client():
    """Get the configured OpenAI client."""
    return OpenAI(api_key=OPENAI_API_KEY)

MEAL_PLAN_SCHEMA = """{
  \"dietary_preference\": \"omnivore|vegetarian|vegan|keto\",
  \"recipes\": [
    {
      \"day\": \"Monday\",
      \"theme\": \"Italian\",
      \"dish_name\": \"Example Dish\",
      \"ingredients\": [
        {\"name\": \"ingredient\", \"quantity\": \"2 cups\"}
      ],
      \"instructions\": \"Brief steps\"
    }
  ],
  \"shopping_list\": [
    {\"name\": \"ingredient\", \"total_quantity\": \"2 cups\", \"used_in\": [\"Example Dish\"]}
  ]
}"""

SYSTEM_PROMPT = (
    "You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.\n\n"
    "STRICT RULES - FOLLOW EXACTLY:\n"
    "1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)\n"
    "2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.\n"
    "3. All ingredients MUST be real, commonly available items\n"
    "4. Follow the dietary preference strictly\n"
    "5. Provide realistic quantities for 2 servings\n\n"
    "Dietary Guidelines:\n"
    "- Omnivore: Any ingredients allowed\n"
    "- Vegetarian: No meat or fish, eggs and dairy OK\n"
    "- Vegan: No animal products at all\n"
    "- Keto: Low carb, high fat, no grains/sugar/starchy vegetables\n\n"
    "Output valid JSON matching this exact structure:\n"
    + MEAL_PLAN_SCHEMA
)

USER_PROMPT_TEMPLATE = (
    "Create a weekly meal plan:\n"
    "- Diet: {dietary_preference}\n"
    "- Theme requests: {theme_requests}\n"
    "- Preferences: {additional_preferences}\n\n"
    "IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday."
)

def build_meal_plan_messages(dietary_preference: str, theme_requests: str, additional_preferences: str):
    """Build messages for the meal plan request."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(
            dietary_preference=dietary_preference,
            theme_requests=theme_requests,
            additional_preferences=additional_preferences
        )}
    ]


## 5. Meal Generation Logic

In [4]:
import json
import time

class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.client = get_client()

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        messages = build_meal_plan_messages(
            dietary_preference=dietary_preference,
            theme_requests=theme_requests if theme_requests else "random diverse cuisines",
            additional_preferences=additional_preferences if additional_preferences else "none"
        )

        result = self.client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=TEMPERATURE,
            messages=messages,
            response_format={"type": "json_object"}
        )

        try:
            content = result.choices[0].message.content
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]

            meal_plan_data = json.loads(content)
            meal_plan = MealPlan(**meal_plan_data)
            generation_time = time.time() - start_time

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(generation_time, 2),
                "error": None
            }

        except json.JSONDecodeError as e:
            return {"success": False, "meal_plan": None, "error": f"JSON parsing error: {str(e)}"}
        except Exception as e:
            return {"success": False, "meal_plan": None, "error": str(e)}


# Initialize generator
generator = MealPlanGenerator()


## 6. Email Functionality

In [5]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


def format_meal_plan_email(meal_plan: dict) -> str:
    """Format meal plan as readable email content (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    # Shopping list FIRST
    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    # Recipes
    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        ingredients = ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)


def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send meal plan via email."""
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_SENDER
        msg['To'] = recipient_email
        msg['Subject'] = "Your Weekly Meal Plan"

        body = format_meal_plan_email(meal_plan)
        msg.attach(MIMEText(body, 'plain'))

        with smtplib.SMTP('smtp.gmail.com', 587) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)

        return {"success": True, "message": f"Email sent to {recipient_email}"}
    except Exception as e:
        return {"success": False, "message": f"Failed to send email: {str(e)}"}

In [6]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Run quick programmatic validation checks (not LLM-based)."""
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error")]}

    meal_plan = meal_plan_result["meal_plan"]
    errors = []

    # Check 1: 7 recipes
    if len(meal_plan["recipes"]) != 7:
        errors.append(f"Expected 7 recipes, got {len(meal_plan['recipes'])}")

    # Check 2: Unique themes
    themes = [r["theme"].lower() for r in meal_plan["recipes"]]
    if len(themes) != len(set(themes)):
        duplicates = [t for t in themes if themes.count(t) > 1]
        errors.append(f"Duplicate themes: {set(duplicates)}")

    # Check 3: 5 ingredients per recipe
    for recipe in meal_plan["recipes"]:
        if len(recipe["ingredients"]) != 5:
            errors.append(f"{recipe['day']}: {len(recipe['ingredients'])} ingredients (expected 5)")

    return {"passed": len(errors) == 0, "errors": errors}

## 7. Gradio Interface

In [7]:
import gradio as gr
import time as pytime


def format_shopping_list(meal_plan: dict) -> str:
    """Format shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients = ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Generate meal plan with streaming output."""
    output = "Generating your meal plan..."
    yield output

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests if theme_requests.strip() else "random",
        additional_preferences=additional_prefs if additional_prefs.strip() else "none"
    )

    if not result["success"]:
        yield f"Error: {result['error']}"
        return

    meal_plan = result["meal_plan"]

    # Show shopping list first
    output = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(meal_plan) + "\n\n---\n\n"
    output += "## Recipes\n\n"
    yield output
    pytime.sleep(0.1)

    # Stream recipes one by one
    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + "\n"
        yield output
        pytime.sleep(0.05)

    # Validation status
    validation = run_basic_validation(result)
    status = "All checks passed" if validation["passed"] else f"Issues: {', '.join(validation['errors'])}"
    output += f"---\n*Generated in {result['generation_time']}s | {status}*"
    yield output


def send_email_ui(email, dietary_pref, theme_requests, additional_prefs):
    """Generate and send meal plan via email."""
    if not email or "@" not in email:
        return "Enter a valid email address."

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests if theme_requests.strip() else "random",
        additional_preferences=additional_prefs if additional_prefs.strip() else "none"
    )

    if not result["success"]:
        return f"Generation failed: {result['error']}"

    email_result = send_meal_plan_email(email, result["meal_plan"])
    return email_result["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, dietary_dropdown, theme_input, additional_input],
        outputs=[email_status]
    )

/home/bella/miniforge3/envs/consulting/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 8. Launch Application

In [8]:
# Launch the Gradio app
# In Colab, this will create a public URL
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6aabd860ba4b7b2f97.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
